# AutoSteer 2.0

## I. Getting started

### Introduction

The AutoSteer 2.0 architecture builds upon components of the YOLO11 framework, incorporating a specially designed head, and predicts the ego-path line directly from the input image. This ego-path representation captures the underlying road geometry and curvature, enabling more robust and accurate steering estimation.

AutoSteer 2.0 processess camera frames in a 2:1 aspect ratio with size 1024px by 512px and predicts ego-path line in camera coordinate system.

### Environment Setup

**If you have done these steps of environment setup in other End-to-End Model Tutorials, please skip this and move forward to the next subsection `3. Download Models`.**

First, we need to clone the repository to your local environment. We will use the official repository from the Autoware Foundation.

In [ ]:
import os

base_dir = os.getcwd()
print(base_dir)

# Clone auto_speed repo, if not yet done
if not os.path.exists("./auto_steer"):
    !git clone https://github.com/autowarefoundation/auto_steer.git

The auto_speed project relies on several key libraries, including PyTorch for model execution and OpenCV for image processing.

We will use pip to install the required dependencies directly from the requirements.txt file provided in the Models directory.

In [ ]:
%pip install --upgrade pip
%pip install -r ./requirements.txt

# Verify the installation
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA availability: {torch.cuda.is_available()}")

### Download Models

You can manually download the model weight files, using the links below. Subsequent tutorial sections assume you already downloaded them and put them inside directory: auto_steer/weights/.

AutoSteer 2.0 model weights - 2:1 aspect ratio, 512px by 1024px input image

### [Link to Download Pytorch Model Weights *.pth](https://drive.google.com/file/d/1-wfPyu7HId7YDSh_T_DkB3Ma1zPc7Fu_/view?usp=drive_link)
### [Link to Download ONNX FP32 Weights *.onnx](https://drive.google.com/file/d/1Rb05CScAfvl0OPRJz8S83Q4BWRG1pEKH/view?usp=drive_link)

You can also use below script block, which uses gdown to download above model weights into such default directory.

In [ ]:
%pip install --upgrade gdown
import os
import gdown

base_dir = os.getcwd()

model_dir = "./weights"
if not os.path.exists(model_dir):
    os.makedirs(model_dir)

metadata = {
    "pth": {
        "url": "https://drive.google.com/file/d/1-wfPyu7HId7YDSh_T_DkB3Ma1zPc7Fu_",
        "filepath": os.path.join(model_dir, "autosteer.pth")
    },
    "onnx": {
        "url": "https://drive.google.com/file/d/1u89PujOd89M-l6t_Cvub3BaQsDFKofuY",
        "filepath": os.path.join(model_dir, "autosteer.onnx")
    }
}


for weight_type in metadata:

    url = metadata[weight_type]["url"]
    filepath = metadata[weight_type]["filepath"]

    if not os.path.exists(filepath):
        print(f"Downloading AutoSteer {weight_type} weights...")
        gdown.download(url, filepath, quiet=False)
    else:
        print(f"AutoSteer {weight_type} weights already exist at {filepath}. Skipping download.")

## II. Test Inference

Pre-requisites: please ensure that you have completed the above **Environment Setup** first.

### 1. Image Inference

The `image_visualization.py` script loads pre-trained model checkpoint, performs a forward pass on the image, and overlays ego path line onto the original frame.

To run this in any environment, we should navigate to the AutoSteer visualization directory so the script's internal relative paths resolve correctly.

Key Script Parameters:
- `-p / --model_checkpoint_path`: location of your `.pth` file containing the trained weights.
- `-i / --input_image_dirpath`: directory containing `.png`, `.jpg`, or `.jpeg` files to process.

In [ ]:
# 1. Path declaration
# Here we use the previously downloaded Pytorch weight file.
# For input and output directories, you can change them to your preferred paths.
# For now we will use the default input and output directories provided in the repository.
import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageDraw

from Models.inference.auto_steer_infer import AutoSteerNetworkInfer

print(base_dir)

CHECKPOINT = metadata["pth"]["filepath"]
INPUT_IMG = os.path.join(
    base_dir,
    "Media/test_6.jpg"
 )


def make_visualization(img, xp, h_vector):
    yp = np.linspace(0, 511, 64, dtype=int)

    e_xp = xp * 1024
    e_h_vector = h_vector
    e_h_vector = (e_h_vector >= 0.5).astype(int)
    e_xp = e_xp * e_h_vector

    # Convert numpy array (assume BGR from cv2) to a PIL Image in RGB
    if isinstance(img, np.ndarray):
        img_rgb = img[..., ::-1] if img.ndim == 3 else img
        pil_img = Image.fromarray(img_rgb.astype(np.uint8))
    else:
        pil_img = img  # already a PIL Image

    draw = ImageDraw.Draw(pil_img)
    radius = 3
    for x, y, h in zip(e_xp, yp, e_h_vector):
        if h == 1:
            draw.ellipse(
                [(x - radius, y - radius), (x + radius, y + radius)],
                fill=(0, 255, 0)  # RGB green, PIL doesn't use BGR like cv2
            )

    plt.imshow(pil_img)
    plt.axis("off")
    plt.show()

checkpoint_abs = os.path.abspath(CHECKPOINT)
input_abs      = os.path.abspath(INPUT_IMG)

color_map = {1: (0, 0, 255), 2: (0, 255, 255),  3: (255, 255, 0) }

model = AutoSteerNetworkInfer(checkpoint_abs)
img = cv2.imread(input_abs)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

xp, h_vector = model.inference(img)
make_visualization(img, xp, h_vector)

### Video Inference

In this subsection, we apply the AutoSpeed model to video streams. The `video_visualization.py` script processes video files frame-by-frame, runs CIPO detection, and writes an annotated output video.

The pipeline supports both **PyTorch** and **ONNX** model formats. It performs inference on each frame and overlays class-colored bounding boxes on detected in-path objects.

Key script parameters:

- `-p / --model_checkpoint_path`: path to model (`.onnx`, `.pt`, or a directory for PyTorch inference).
- `-i / --video_filepath`: path to the source video file.
- `-o / --output_file`: output path prefix for the annotated video (script appends `.avi`).
- `-v / --vis`: optional flag to preview frames during processing.

In [ ]:
import subprocess
from IPython.display import Video, display

# 1. Path declaration
# Similarly as image inference above:
# - Pytorch weight file (ONXX okay too).
# - Input/Output paths can be changed to preferred.
# - Using defaults for now.
MODEL_PATH = metadata["pth"]["filepath"]
INPUT_PATH = os.path.join(
    base_dir,
    "Media/highway_1024_512.mp4"
 )
OUTPUT_BASENAME = os.path.join(
    base_dir,
    "./results/highway_1024_512"
 )

# 2. Navigate to visualization directory and execute the script
!mkdir results
!cd ./Models/visualizations/ && \
python3 video_visualization.py \
    -a {os.path.abspath(MODEL_PATH)} \
    -i {os.path.abspath(INPUT_PATH)} \
    -o {os.path.abspath(OUTPUT_BASENAME)}

print(f"Saved video: {os.path.abspath(OUTPUT_BASENAME)}.avi")

Play the generated output video

In [ ]:
import os
from IPython.display import Video

# Display with specific dimensions and embedded data
print(f"Output video: {os.path.abspath(OUTPUT_BASENAME)}.avi")
Video(os.path.join(os.path.abspath(OUTPUT_BASENAME), "avi"), width=640, height=360, embed=True)

## III. Model Training

### Convert data

AutoSteer 2.0 model is trained on three datasets combined  [TUSimple](https://www.kaggle.com/datasets/manideep1108/tusimple), [OpenLane](https://github.com/OpenDriveLab/OpenLane) and [CurveLanes](https://github.com/SoulmateB/CurveLanes). In order to convert every dataset download the dataset into corresponding directory.

#### TUSimple

In [ ]:
import os

INPUT_DATASET_DIR = "<INPUT_DATASET_DIR>"
OUTPUT_DATASET_DIR = os.path.join(os.path.abspath(INPUT_DATASET_DIR), "_AutoSteer_2.0")

!cd ./Models/data_parsing/TuSimple/ && \
python3 converter.py -i {os.path.abspath(INPUT_DATASET_DIR)} -o {OUTPUT_DATASET_DIR}

#### OpenLane

In [ ]:
import os

INPUT_DATASET_DIR = "<INPUT_DATASET_DIR>"
OUTPUT_DATASET_DIR = os.path.join(os.path.abspath(INPUT_DATASET_DIR), "_AutoSteer_2.0")

!cd ./Models/data_parsing/OpenLane/ && \
python3 converter.py -i {os.path.abspath(INPUT_DATASET_DIR)} -o {OUTPUT_DATASET_DIR}

#### CurveLanes

In [ ]:
import os

INPUT_DATASET_DIR = "<INPUT_DATASET_DIR>"
OUTPUT_DATASET_DIR = os.path.join(os.path.abspath(INPUT_DATASET_DIR), "_AutoSteer_2.0")

!cd ./Models/data_parsing/CurveLanes/ && \
python3 converter.py -i {os.path.abspath(INPUT_DATASET_DIR)} -o {OUTPUT_DATASET_DIR}

After all three datasets are converted combine all together in common directory example
```
AutoSteer_1.0
    ├── images/
    │     ├── train/
    │     └── val/
    └── labels/
          ├── train/
          └── val/
```

### Run training

In [ ]:
import os

DATASET_DIR = "AutoSteer_1.0"

!cd ./Models/training/ && \
python3 auto_steer_trainer.py --dataset {os.path.abspath(DATASET_DIR)}